<a href="https://colab.research.google.com/github/Dina-me10/CeraGPT/blob/main/Cera_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# [9]
!pip install transformers torch bitsandbytes datasets peft

In [28]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset

from peft import LoraConfig, get_peft_model

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',      # GPU bo'lsa GPU'ga, bo'lmasa CPU'ga joylaydi
    dtype=torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Before Fine-tuning
prompt = "My skin is very dry in winter. What should I use?"
# prompt = "For very dry skin in winter, the best skincare routine is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)  # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)
first_block = model.model.layers[0]
print('first_block:', first_block)
print('=======')
print(first_block.self_attn)
print('=======')
print(model.config)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My skin is very dry in winter. What should I use?
3. What is the recommended use for the Bioderma Lotion for dry skin?
4. How can I make the Bioderma Lotion smell pleasant for my taste?
5. How can I apply the Bioderma Lotion for dry skin?
6. What is the recommended use for the Bioderma Hydro Plus Cream for dry
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), e

In [30]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272


dataset library | load_dataset
##instruction tuning

In [31]:


dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [32]:
skin_data = dataset.filter(
    lambda x: "skin" in x["instruction"].lower()
)

print("Skincare misollari soni:", len(skin_data))
print(skin_data[0])

Skincare misollari soni: 32
{'output': "Hello, I'm sorry to hear that you're having trouble logging in to your account. Can you please tell me more about the issue you're experiencing? Are you receiving any error messages or is it not accepting your credentials? In the meantime, you may want to try resetting your password or clearing your cache and cookies. Please provide me with a bit more information so that I can better assist you in resolving this issue.", 'input': "I'm having difficulty logging in to my account.", 'instruction': 'Generate an appropriate response to a customer who is asking for technical support.'}


In [33]:
def generate_prompt(example):
    instruction = example['instruction']
    input_text = example['input']
    output_text = example['output']

    if input_text:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output_text}"
        )

# generate_prompt(dataset[0])

def formatting_func(example):
    return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.',
 'text': '### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive 

In [34]:
print(dataset[0]['text'])

### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.


In [35]:
dataset = dataset.shuffle(seed=42)

In [36]:
dataset = dataset.select(range(7000))

In [37]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})

In [38]:
# Full Fine-tuning  => modelning hammasini o'qitish
# Cheap Fine-tuning => faqat kichik qismini o'qitish
# PEFT => Parameter Efficient Fine Tuning
# OOM  => Out of Memory (xotira yetmay qolishi)


lora_config = LoraConfig(
    r=8,  # rank
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

In [39]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
